In [2]:
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
import argostranslate.package, argostranslate.translate
import re, os, json, time

In [3]:
dados = pd.read_csv("./completoComLetras.csv")

In [6]:
DetectorFactory.seed = 0   # resultado estável

def idioma(texto):
    try:
        return detect(texto[:2000])
    except Exception:
        return None

#unicas = dados.drop_duplicates("track_id")[["track_id", "letra"]].copy()
dados["idioma"] = dados["letra"].fillna("").map(idioma)
print(dados["idioma"].value_counts())

idioma
en    850
es    682
pt    332
ja    289
ko      3
sw      2
fr      1
tl      1
Name: count, dtype: int64


In [ ]:
# pip install transformers sentencepiece torch deep-translator
# ---------- configuração ----------
CACHE = "letras_en.jsonl"                 # só guarda traduções aprovadas
ORDEM = ["nllb", "google"]                # do pior ao melhor; cada motor só pega o que sobrou
NLLB_MODEL = "facebook/nllb-200-distilled-600M"
NLLB_BATCH = 16
GOOGLE_PAUSA = 1.0
GOOGLE_MAX = 4500                         # limite do deep-translator é ~5000 por chamada

# idioma detectado -> (código NLLB, código Google)
LANGS = {
    "es": ("spa_Latn", "es"),
    "pt": ("por_Latn", "pt"),
    "ja": ("jpn_Jpan", "ja"),
    "ko": ("kor_Hang", "ko"),
}
# sw/tl/fr (raros) quase sempre são detecção errada de letras es/pt com gíria:
# não entram no NLLB; o Google tenta com detecção automática.
GOOGLE_AUTO = {"sw", "tl", "fr"}

# ---------- 0. base de letras únicas ----------
unicas = (dados.drop_duplicates("track_id")[["track_id", "idioma", "letra"]]
          .assign(letra=lambda d: d["letra"].fillna("").astype(str)))
unicas = unicas[unicas["letra"].str.strip() != ""]

# ---------- 1. cache ----------
def le_cache():
    if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
        return pd.read_json(CACHE, lines=True)
    return pd.DataFrame(columns=["track_id", "letra_en", "motor"])

feitos = set(le_cache()["track_id"])

def grava(f, track_id, idioma, letra_en, motor):
    f.write(json.dumps({"track_id": track_id, "idioma": idioma,
                        "letra_en": letra_en, "motor": motor}, ensure_ascii=False) + "\n")
    f.flush()

# ---------- 2. checagem de qualidade ----------
_CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]")

def traducao_ok(orig, trad, idioma):
    if not trad or not trad.strip():
        return False
    lo = [l.strip().lower() for l in orig.split("\n") if l.strip()]
    lt = [l.strip().lower() for l in trad.split("\n") if l.strip()]
    if not lt:
        return False
    razao = len(trad) / max(len(orig), 1)
    if idioma in ("ja", "ko"):
        if len(_CJK.findall(trad)) / max(len(trad), 1) > 0.05:   # ainda em kana/hangul
            return False
        return 0.3 < razao < 6
    iguais = sum(a == b for a, b in zip(lo, lt)) / max(len(lo), 1)
    return iguais < 0.6 and 0.3 < razao < 3                      # não devolveu quase igual

# ---------- 3. motor NLLB (carrega só se for usado) ----------
_nllb = {}

def traduz_nllb(letra, idioma):
    import torch
    if idioma not in LANGS:
        return None
    if not _nllb:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _nllb["tok"] = AutoTokenizer.from_pretrained(NLLB_MODEL)
        #_nllb["mod"] = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL).to(dev).eval()
        _nllb["mod"] = AutoModelForSeq2SeqLM.from_pretrained(
                        NLLB_MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32
                ).to(dev).eval()
        _nllb["dev"] = dev
        print("NLLB carregado em", dev)
    tok, mod, dev = _nllb["tok"], _nllb["mod"], _nllb["dev"]
    tok.src_lang = LANGS[idioma][0]
    eng = tok.convert_tokens_to_ids("eng_Latn")

    linhas = letra.split("\n")
    unicas_l = list(dict.fromkeys(l for l in linhas if l.strip()))   # refrão repetido traduz 1 vez
    mapa = {}
    for i in range(0, len(unicas_l), NLLB_BATCH):
        lote = unicas_l[i:i + NLLB_BATCH]
        enc = tok(lote, return_tensors="pt", padding=True, truncation=True, max_length=200).to(dev)
        with torch.no_grad():
            out = mod.generate(**enc, forced_bos_token_id=eng, max_new_tokens=200, num_beams=4)
        for o, t in zip(lote, tok.batch_decode(out, skip_special_tokens=True)):
            mapa[o] = t
    return "\n".join(mapa.get(l, "") if l.strip() else "" for l in linhas)

# ---------- 4. motor Google (deep-translator) ----------
def traduz_google(letra, idioma):
    from deep_translator import GoogleTranslator
    src = "auto" if idioma in GOOGLE_AUTO or idioma not in LANGS else LANGS[idioma][1]
    gt = GoogleTranslator(source=src, target="en")

    linhas = letra.split("\n")
    blocos, atual, tam = [], [], 0
    for l in linhas:                                   # agrupa linhas sem passar do limite
        if tam + len(l) + 1 > GOOGLE_MAX and atual:
            blocos.append(atual); atual, tam = [], 0
        atual.append(l); tam += len(l) + 1
    if atual:
        blocos.append(atual)

    saida = []
    for b in blocos:
        texto = "\n".join(b)
        if not texto.strip():
            saida.append(texto); continue
        for tent in range(3):                          # retry com espera
            try:
                saida.append(gt.translate(texto) or "")
                break
            except Exception:
                time.sleep(3 * (tent + 1))
        else:
            return None                                # bloco falhou: faixa continua pendente
        time.sleep(GOOGLE_PAUSA)
    return "\n".join(saida)

MOTORES = {"nllb": traduz_nllb, "google": traduz_google}

# ---------- 5. cascata ----------
# inglês não precisa de tradução: entra direto
with open(CACHE, "a", encoding="utf-8") as f:
    for r in unicas[(unicas["idioma"] == "en") & (~unicas["track_id"].isin(feitos))].itertuples():
        grava(f, r.track_id, "en", r.letra, "original")
        feitos.add(r.track_id)

for motor in ORDEM:
    pend = unicas[~unicas["track_id"].isin(feitos)]
    print(f"\n[{motor}] pendentes: {len(pend)}")
    if pend.empty:
        break
    ok = 0
    with open(CACHE, "a", encoding="utf-8") as f:
        for r in pend.itertuples():
            try:
                trad = MOTORES[motor](r.letra, r.idioma)
            except Exception as e:
                print("erro", motor, r.track_id, repr(e)[:100])
                continue
            if trad is not None and traducao_ok(r.letra, trad, r.idioma):
                grava(f, r.track_id, r.idioma, trad, motor)
                feitos.add(r.track_id)
                ok += 1
    print(f"[{motor}] aprovadas: {ok}")

# ---------- 6. merge no dados ----------
en = le_cache().drop_duplicates("track_id")[["track_id", "letra_en", "motor"]] \
               .rename(columns={"motor": "motor_traducao"})
dados = dados.drop(columns=[c for c in ["letra_en", "motor_traducao"] if c in dados.columns]) \
             .merge(en, on="track_id", how="left")

# ---------- 7. relatório ----------
print("\nTraduções por motor (tracks únicas):")
print(dados.drop_duplicates("track_id")["motor_traducao"].value_counts(dropna=False))

sem = dados[dados["letra_en"].isna() & (dados["letra"].fillna("").str.strip() != "")] \
        .drop_duplicates("track_id")
print(len(sem), "tracks sem tradução aprovada")
display(sem[["track_id", "idioma", "title", "artists"]])


[nllb] pendentes: 592


HTTP Error 429 thrown while requesting GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/tree/main/additional_chat_templates
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/tree/main/additional_chat_templates
Retrying in 4s [Retry 3/5].
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 8635.84it/s]


NLLB carregado em cpu


[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

In [ ]:
# instala os pacotes X -> en (só na primeira vez)
argostranslate.package.update_package_index()
disponiveis = argostranslate.package.get_available_packages()
for src in ["pt", "es", "ko", "ja", "sw", "fr", "tl"]:
    pkg = next((p for p in disponiveis if p.from_code == src and p.to_code == "en"), None)
    if pkg:
        argostranslate.package.install_from_path(pkg.download())
    else:
        print("sem pacote para", src)

def traduz(letra, src):
    linhas = letra.split("\n")
    out = [argostranslate.translate.translate(l, src, "en") if l.strip() else "" for l in linhas]
    return "\n".join(out)

CACHE = "letras_en.jsonl"
feitos = set()
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

with open(CACHE, "a", encoding="utf-8") as f:
    for row in unicas.itertuples():
        if row.track_id in feitos or not row.letra:
            continue
        if row.idioma == "en":
            letra_en = row.letra
        elif row.idioma in ("pt", "es", "ko", "ja", "sw", "fr", "tl"):
            letra_en = traduz(row.letra, row.idioma)
        else:
            continue   # idioma não detectado ou fora da lista: revise à mão
        f.write(json.dumps({"track_id": row.track_id, "idioma": row.idioma,
                            "letra_en": letra_en}, ensure_ascii=False) + "\n")
        f.flush()

en = pd.read_json(CACHE, lines=True)
dados = dados.drop(columns=[c for c in ["letra_en", "idioma"] if c in dados.columns]) \
             .merge(en[["track_id", "idioma", "letra_en"]], on="track_id", how="left")

2026-11-05 17:50:13 WARNING: Language es package default expects mwt, which has been added
2026-11-05 17:50:25 WARNING: Language pt package default expects mwt, which has been added
